<h2>Detect and remove duplicate records by employing hashing and record-matching techniques, ensuring data integrity while maintaining unique entries across large and complex datasets.</h2>

In [1]:
import hashlib
import pandas as pd

In [2]:
data = {
    'transaction_id': [101, 102, 101, 103, 104],
    'customer_name': [' Alice Smith ', 'Bob Jones', 'Alice Smith', 'Alice Smith ', 'Bob Jones'],
    'email': ['alice@example.com', 'bob@example.com', 'alice@example.com', 'ALICE@EXAMPLE.COM', 'bob@example.com'],
    'amount': [250.0, 100.0, 250.0, 250.0, 100.0]
}

In [3]:
df = pd.DataFrame(data)
print("Original Raw Data:")
print(df)

Original Raw Data:
   transaction_id  customer_name              email  amount
0             101   Alice Smith   alice@example.com   250.0
1             102      Bob Jones    bob@example.com   100.0
2             101    Alice Smith  alice@example.com   250.0
3             103   Alice Smith   ALICE@EXAMPLE.COM   250.0
4             104      Bob Jones    bob@example.com   100.0


In [4]:
df['clean_name'] = df['customer_name'].astype(str).str.strip().str.lower()
df['clean_email'] = df['email'].astype(str).str.strip().str.lower()

In [5]:
print("Data After Standardization:")
print(df[['clean_name', 'clean_email', 'amount']])

Data After Standardization:
    clean_name        clean_email  amount
0  alice smith  alice@example.com   250.0
1    bob jones    bob@example.com   100.0
2  alice smith  alice@example.com   250.0
3  alice smith  alice@example.com   250.0
4    bob jones    bob@example.com   100.0


In [7]:
def generate_row_hash(row):
    combined_string = f"{row['clean_name']}|{row['clean_email']}|{row['amount']}"
    return hashlib.md5(combined_string.encode('utf-8')).hexdigest()

In [8]:
df['record_hash'] = df.apply(generate_row_hash, axis=1)

In [9]:
print("Generated Hash Keys for Each Record:")
print(df[['clean_name', 'clean_email', 'amount', 'record_hash']])

Generated Hash Keys for Each Record:
    clean_name        clean_email  amount                       record_hash
0  alice smith  alice@example.com   250.0  7266a6ea4880f0f68c8d175580fd772f
1    bob jones    bob@example.com   100.0  328f5d7f0f14716013dba0a8d6781c07
2  alice smith  alice@example.com   250.0  7266a6ea4880f0f68c8d175580fd772f
3  alice smith  alice@example.com   250.0  7266a6ea4880f0f68c8d175580fd772f
4    bob jones    bob@example.com   100.0  328f5d7f0f14716013dba0a8d6781c07


In [10]:
df_unique = df.drop_duplicates(subset=['record_hash'], keep='first').copy()

In [11]:
df_cleaned = df_unique.drop(columns=['clean_name', 'clean_email', 'record_hash']).reset_index(drop=True)

In [12]:
print("Final Cleaned Dataset (Unique Entries Only):")
print(df_cleaned)

Final Cleaned Dataset (Unique Entries Only):
   transaction_id  customer_name              email  amount
0             101   Alice Smith   alice@example.com   250.0
1             102      Bob Jones    bob@example.com   100.0
